[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn
import math

In [51]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.Wq = nn.Parameter(torch.randn(d_model, d_model))
        self.Wk = nn.Parameter(torch.randn(d_model, d_model))
        self.Wv = nn.Parameter(torch.randn(d_model, d_model))
        self.d_model = torch.tensor(d_model)
        self.num_heads = torch.tensor(num_heads)

    def forward(self, x_q, x_kv):
        Q = x_q @ self.Wq # b , nq , d
        K = x_kv @  self.Wk # b , nk , d
        V = x_kv @  self.Wv # b , nk , d

        Q_ = torch.chunk(Q, self.num_heads, dim=-1)
        K_ = torch.chunk(K, self.num_heads, dim=-1)
        V_ = torch.chunk(K, self.num_heads, dim=-1)

        z_list = []
        for i in range(self.num_heads) :
          q, k, v = Q_[i] , K_[i] , V_[i]
          k_T = torch.permute(k , dims = (0,2,1))
          atten = torch.softmax(q @ k_T/ torch.sqrt(self.d_model), dim = -1) # b , nq , nk
          z = atten @ v # b , nq , d / h
          z_list.append(z)

        z_out = torch.concat(z_list, dim = -1 )
        return z_out




In [52]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

Output: torch.Size([2, 6, 64])


In [53]:

# ✅ SUBMIT
from torch_judge import check
check('cross_attention')


🧪 Testing: Multi-Head Cross-Attention (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (4.5ms)
  ✅ [2/4] Q and KV different lengths (2.6ms)
  ✅ [3/4] No causal mask — all KV affects all Q (7.5ms)
  ✅ [4/4] Gradient flow (5.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (20.5ms total)
  Progress saved. Run status() to see your dashboard.

